<a href="https://colab.research.google.com/github/wandb/examples/blob/master/colabs/pytorch/Simple_PyTorch_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<!--- @wandbcode{pytorch-video} -->

<img src="http://wandb.me/logo-im-png" width="400" alt="Weights & Biases" />

<!--- @wandbcode{pytorch-video} -->

# 🔥 = W&B ➕ PyTorch
Use [Weights & Biases](https://wandb.com) for machine learning experiment tracking, dataset versioning, and project collaboration.

<div><img /></div>

<img src="https://wandb.me/mini-diagram" width="650" alt="Weights & Biases" />

<div><img /></div>




## What this notebook covers:

We show you how to integrate Weights & Biases with your PyTorch code to add experiment tracking to your pipeline.

## The resulting interactive W&B dashboard will look like:
![](https://i.imgur.com/z8TK2Et.png)

## In pseudocode, what we'll do is:
```python
# import the library
import wandb

# start a new experiment
wandb.init(project="new-sota-model")

# capture a dictionary of hyperparameters with config
wandb.config = {"learning_rate": 0.001, "epochs": 100, "batch_size": 128}

# set up model and data
model, dataloader = get_model(), get_data()

# optional: track gradients
wandb.watch(model)

for batch in dataloader:
  metrics = model.training_step()
  # log metrics inside your training loop to visualize model performance
  wandb.log(metrics)

# optional: save model at the end
model.to_onnx()
wandb.save("model.onnx")
```



## Follow along with a [video tutorial](http://wandb.me/pytorch-video)!
**Note**: Sections starting with _Step_ are all you need to integrate W&B in an existing pipeline. The rest just loads data and defines a model.

# 🚀 Install, Import, and Log In

In [1]:
import pandas as pd
import time
import torch.optim as optim
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from matplotlib import pyplot as plt

In [2]:
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(hash("improves reproducibility") % 2**32 - 1)
torch.manual_seed(hash("by removing stochasticity") % 2**32 - 1)
torch.cuda.manual_seed_all(hash("so runs are repeatable") % 2**32 - 1)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [3]:
!pip install lightning

In [4]:
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torch.nn as nn
import os
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from matplotlib import pyplot as plt
import lightning.pytorch as pl
import torch.nn.functional as F
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from torchvision.models import resnet50, ResNet50_Weights

### 0️⃣ Step 0: Install W&B

To get started, we'll need to get the library.
`wandb` is easily installed using `pip`.

In [5]:
!pip install wandb -Uq

### 1️⃣ Step 1: Import W&B and Login

In order to log data to our web service,
you'll need to log in.

If this is your first time using W&B,
you'll need to sign up for a free account at the link that appears.

In [6]:
import wandb

wandb.login()

<IPython.core.display.Javascript object>

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

# 👩‍🔬 Define the Experiment and Pipeline

## 2️⃣ Step 2: Track metadata and hyperparameters with `wandb.init`

Programmatically, the first thing we do is define our experiment:
what are the hyperparameters? what metadata is associated with this run?

It's a pretty common workflow to store this information in a `config` dictionary
(or similar object)
and then access it as needed.

For this example, we're only letting a few hyperparameters vary
and hand-coding the rest.
But any part of your model can be part of the `config`!

We also include some metadata: we're using the MNIST dataset and a convolutional
architecture. If we later work with, say,
fully-connected architectures on CIFAR in the same project,
this will help us separate our runs.

In [8]:
config = dict(
    epochs=60,
    batch_size=10,
    learning_rate=0.001,
    dataset="aug_224",
    architecture="ResNet")

Now, let's define the overall pipeline,
which is pretty typical for model-training:

1. we first `make` a model, plus associated data and optimizer, then
2. we `train` the model accordingly and finally
3. `test` it to see how training went.

We'll implement these functions below.

In [9]:
def model_pipeline(hyperparameters):

    # tell wandb to get started
    with wandb.init(project="646_project", config=hyperparameters):
      # access all HPs through wandb.config, so logging matches execution!
      config = wandb.config

      # make the model, data, and optimization problem
      model, train_loader, val_loader, test_loader, criterion, optimizer = make(config)
      print(model)

      # and use them to train the model
      train(model, train_loader, val_loader, criterion, optimizer, config)

      # and test its final performance
      test(model, test_loader, config)

    return model

The only difference here from a standard pipeline
is that it all occurs inside the context of `wandb.init`.
Calling this function sets up a line of communication
between your code and our servers.

Passing the `config` dictionary to `wandb.init`
immediately logs all that information to us,
so you'll always know what hyperparameter values
you set your experiment to use.

To ensure the values you chose and logged are always the ones that get used
in your model, we recommend using the `wandb.config` copy of your object.
Check the definition of `make` below to see some examples.

> *Side Note*: We take care to run our code in separate processes,
so that any issues on our end
(e.g. a giant sea monster attacks our data centers)
don't crash your code.
Once the issue is resolved (e.g. the Kraken returns to the deep)
you can log the data with `wandb sync`.

In [10]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [11]:
class CephaDataset(Dataset):
    def __init__(self, base_folder='/content/drive/MyDrive/data/aug_img_224', split='train', one_channel=True):
        self.one_channel = one_channel
        self.images_list = []
        self.coordinates_list = []
        # self.name = []
        def extract_labels_from_txt(path):
            with open(path, "r") as f:
                # only first 19 are actual coords in dataset label files
                coords_raw = f.readlines()[:19]
                coords_raw = [tuple([int(float(s)) for s in t.split(",")]) for t in coords_raw]
                return coords_raw

        folder = os.path.join(base_folder, split)
        for file in os.listdir(folder):
            path = os.path.join(folder, file)
            if path.endswith(".png"):
                if one_channel:
                    image = Image.open(path).convert('L')
                else:
                    image = Image.open(path)
                # image = Image.open(path).convert('L')
                label_path = path.replace('.png', '.txt')
                label = extract_labels_from_txt(label_path)
                self.images_list.append(image)
                self.coordinates_list.append(label)
                # self.name.append(path)

    def __len__(self):
        return len(self.images_list)
    
    def __getitem__(self, idx):
        # convert to numpy array
        # if directly convert to tensor, we cannot plot it because image has size (1, 160, 160)
        image = self.images_list[idx]
        image = np.array(image).astype(np.float32)
        if not self.one_channel:
            image = np.moveaxis(image, 2, 0)
        coordinates = self.coordinates_list[idx]
        coordinates = np.array(coordinates).astype(np.float32).flatten()
        return image, coordinates

In [12]:
def make(config):
    # Make the data
    valset = CephaDataset(base_folder='/content/drive/MyDrive/data/aug_img_224', split='test1', one_channel=False)
    trainset = CephaDataset(base_folder='/content/drive/MyDrive/data/aug_img_224', split='train', one_channel=False)
    testset = CephaDataset(base_folder='/content/drive/MyDrive/data/aug_img_224', split='test2', one_channel=False)

    train_loader = DataLoader(trainset, config.batch_size, shuffle=True)
    val_loader = DataLoader(valset, config.batch_size, shuffle=False)
    test_loader = DataLoader(testset, config.batch_size, shuffle=True)

    # Make the model
    model = ResNet50Model().to(device)

    # Make the loss and optimizer
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=config.learning_rate)
    
    return model, train_loader, val_loader, test_loader, criterion, optimizer

In [13]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

# 📡 Define the Data Loading and Model

Now, we need to specify how the data is loaded and what the model looks like.

This part is very important, but it's
no different from what it would be without `wandb`,
so we won't dwell on it.

Defining the model is normally the fun part!

But nothing changes with `wandb`,
so we're gonna stick with a standard ConvNet architecture.

Don't be afraid to mess around with this and try some experiments --
all your results will be logged on [wandb.ai](https://wandb.ai)!



In [14]:
class ResNet50Model(pl.LightningModule):
    def __init__(self, pretrained=False, in_channels = 3, outut_size = 38, lr=3e-4):
        super(ResNet50Model, self).__init__()
        self.in_channels = in_channels
        self.output_size = outut_size
        self.lr = lr
        self.model = resnet50(pretrained=False)
        self.model.fc = nn.Sequential(
            # nn.Linear(self.model.fc.in_features, 128),
            # nn.ReLU(),
            # nn.Linear(128, 128),
            # nn.BatchNorm1d(128),
            # nn.ReLU(),
            # nn.Linear(128, self.output_size)
            nn.Linear(self.model.fc.in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, self.output_size)
        )
        self.loss_fn = nn.MSELoss()
    
    def forward(self, x):
        return self.model(x)
    
    def configure_optimizers(self):
        # optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr)
        # optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2)
        # return [optimizer], [scheduler]
        # optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        optimizer = torch.optim.SGD(self.parameters(), lr=self.lr)
        return optimizer
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        preds = self.model(x)
        # loss = self.loss_fn(preds, y)
        loss = F.mse_loss(preds, y)
        # self.train_acc(torch.argmax(preds, dim=1), y)
        # self.log('train_loss', loss.item(), on_epoch=True)
        # self.log('train_acc', self.train_acc, on_epoch=True)
        self.log('train_loss', loss)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        preds = self.model(x)
        # val_loss = self.loss_fn(preds, y)
        val_loss = F.mse_loss(preds, y)
        # self.val_acc(torch.argmax(preds, dim=1), y)
        # self.log('val_loss', loss.item(), on_epoch=True)
        # self.log('val_acc', self.val_acc, on_epoch=True)
        self.log('val_loss', val_loss)
        return {'val_loss': val_loss}

# 👟 Define Training Logic

Moving on in our `model_pipeline`, it's time to specify how we `train`.

Two `wandb` functions come into play here: `watch` and `log`.

### 3️⃣ Step 3. Track gradients with `wandb.watch` and everything else with `wandb.log`

`wandb.watch` will log the gradients and the parameters of your model,
every `log_freq` steps of training.

All you need to do is call it before you start training.

The rest of the training code remains the same:
we iterate over epochs and batches,
running forward and backward passes
and applying our `optimizer`.

In [15]:
def train(model, train_loader, val_loader, criterion, optimizer, confi):
    # Tell wandb to watch what the model gets up to: gradients, weights, and more!
    wandb.watch(model, criterion, log="all", log_freq=10)

    # Run training and track with wandb
    model = model.to(device)
    model.eval()

    start_time = time.time()
    #liveloss = PlotLosses()
    total_batches = len(train_loader) * config['epochs']
    example_ct = 0  # number of examples seen
    batch_ct = 0

    for epoch in range(0, config['epochs']):
        #logs = {}
        model.train()
        train_loss = 0
        for (batch_id, (imgs, labels)) in enumerate(train_loader):
            imgs = imgs.to(device)
            labels = labels.to(device)

            predicted = model(imgs.float())
            optimizer.zero_grad()

            loss = criterion(predicted, labels.view(config['batch_size'], 38).float())

            loss.backward()
            optimizer.step()

            example_ct +=  len(imgs)
            batch_ct += 1

            # Report metrics every 10th batch
            if (batch_ct % 10) == 0:
                train_log(loss, example_ct, epoch)

        #train_loss /= len(train_loader)
        #logs['loss'] = train_loss

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for (imgs, labels) in val_loader:
                imgs = imgs.to(device)
                labels = labels.to(device)

                predicted = model(imgs.float())
                loss = criterion(predicted, labels.view(config['batch_size'], 38).float())
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_log(val_loss, example_ct, epoch)
        
        #val_loss /= len(val_loader)
        #logs['val loss'] = val_loss
        #liveloss.update(logs)
        #liveloss.send()

    print("Total time elapsed: {:.2f} seconds".format(time.time() - start_time))

The only difference is in the logging code:
where previously you might have reported metrics by printing to the terminal,
now you pass the same information to `wandb.log`.

`wandb.log` expects a dictionary with strings as keys.
These strings identify the objects being logged, which make up the values.
You can also optionally log which `step` of training you're on.

> *Side Note*: I like to use the number of examples the model has seen,
since this makes for easier comparison across batch sizes,
but you can use raw steps or batch count. For longer training runs, it can also make sense to log by `epoch`.

In [16]:
def train_log(loss, example_ct, epoch):
    # Where the magic happens
    wandb.log({"epoch": epoch, "training loss": loss}, step=example_ct)
    print(f"Training loss after {str(example_ct).zfill(5)} examples: {loss:.3f}")

In [17]:
def val_log(loss, example_ct, epoch):
    # Where the magic happens
    wandb.log({"epoch": epoch, "validation loss": loss}, step=example_ct)
    print(f"Validation loss after epoch {epoch}: {loss:.3f}")

# 🧪 Define Testing Logic

Once the model is done training, we want to test it:
run it against some fresh data from production, perhaps,
or apply it to some hand-curated "hard examples".



#### 4️⃣ Optional Step 4: Call `wandb.save`

This is also a great time to save the model's architecture
and final parameters to disk.
For maximum compatibility, we'll `export` our model in the
[Open Neural Network eXchange (ONNX) format](https://onnx.ai/).

Passing that filename to `wandb.save` ensures that the model parameters
are saved to W&B's servers: no more losing track of which `.h5` or `.pb`
corresponds to which training runs!

For more advanced `wandb` features for storing, versioning, and distributing
models, check out our [Artifacts tools](https://www.wandb.com/artifacts).

In [18]:
def accuracy_function(predicted_coordinates, true_coordinates, threshold):
    predicted_coordinates = np.array(predicted_coordinates)
    true_coordinates = np.array(true_coordinates)
    
    dist = np.sqrt(np.sum((predicted_coordinates-true_coordinates)**2,axis=1))
    correct_predictions = np.sum(dist <= threshold)
    total_predictions = len(predicted_coordinates)
    accuracy = correct_predictions / total_predictions
    return accuracy

In [19]:
def group(lst, n):
    return list(zip(*[lst[i::n] for i in range(n)]) )


In [20]:
def test(model, test_loader, config):
    model.eval()
    with torch.no_grad():
        true_labels = []
        predicted_labels = []
        for batch_idx, (imgs, labels) in enumerate(test_loader):
          imgs = imgs.to(device)
          labels = labels.to(device)
          predictions = model(imgs.float()).data.cpu().numpy()
          for i in range(config['batch_size']):
            true_labels += labels[i].data.cpu().numpy().tolist()
            prediction = group(predictions[i], 2)
            predicted_labels += prediction
        
        true_labels = group(true_labels, 2)
        test_acc = accuracy_function(predicted_labels, true_labels, threshold=10)
        print(f"Percentage of predicted points considered to be accurate within a threshold deviation of 10: {test_acc}")
        wandb.log({"test_accuracy": test_acc})

    # Save the model in the exchangeable ONNX format
    torch.onnx.export(model, imgs, "model.onnx")
    wandb.save("model.onnx")

# 🏃‍♀️ Run training and watch your metrics live on wandb.ai!

Now that we've defined the whole pipeline and slipped in
those few lines of W&B code,
we're ready to run our fully-tracked experiment.

We'll report a few links to you:
our documentation,
the Project page, which organizes all the runs in a project, and
the Run page, where this run's results will be stored.

Navigate to the Run page and check out these tabs:

1. **Charts**, where the model gradients, parameter values, and loss are logged throughout training
2. **System**, which contains a variety of system metrics, including Disk I/O utilization, CPU and GPU metrics (watch that temperature soar 🔥), and more
3. **Logs**, which has a copy of anything pushed to standard out during training
4. **Files**, where, once training is complete, you can click on the `model.onnx` to view our network with the [Netron model viewer](https://github.com/lutzroeder/netron).

Once the run in finished
(i.e. the `with wandb.init` block is exited),
we'll also print a summary of the results in the cell output.

In [22]:
!pip install onnx

In [ ]:
# Build, train and analyze the model with the pipeline
model = model_pipeline(config)

# 🧹 Test Hyperparameters with Sweeps

We only looked at a single set of hyperparameters in this example.
But an important part of most ML workflows is iterating over
a number of hyperparameters.

You can use Weights & Biases Sweeps to automate hyperparameter testing and explore the space of possible models and optimization strategies.

## [Check out Hyperparameter Optimization in PyTorch using W&B Sweeps $\rightarrow$](http://wandb.me/sweeps-colab)

Running a hyperparameter sweep with Weights & Biases is very easy. There are just 3 simple steps:

1. **Define the sweep:** We do this by creating a dictionary or a [YAML file](https://docs.wandb.com/library/sweeps/configuration) that specifies the parameters to search through, the search strategy, the optimization metric et all.

2. **Initialize the sweep:** 
`sweep_id = wandb.sweep(sweep_config)`

3. **Run the sweep agent:** 
`wandb.agent(sweep_id, function=train)`

And voila! That's all there is to running a hyperparameter sweep!
<img src="https://imgur.com/UiQKg0L.png" alt="Weights & Biases" />


# 🖼️ Example Gallery

See examples of projects tracked and visualized with W&B in our [Gallery →](https://app.wandb.ai/gallery)

# 🤓 Advanced Setup
1. [Environment variables](https://docs.wandb.com/library/environment-variables): Set API keys in environment variables so you can run training on a managed cluster.
2. [Offline mode](https://docs.wandb.com/library/technical-faq#can-i-run-wandb-offline): Use `dryrun` mode to train offline and sync results later.
3. [On-prem](https://docs.wandb.com/self-hosted): Install W&B in a private cloud or air-gapped servers in your own infrastructure. We have local installations for everyone from academics to enterprise teams.
4. [Sweeps](https://docs.wandb.com/sweeps): Set up hyperparameter search quickly with our lightweight tool for tuning.